In [1]:
!pip install adjustText

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import re
from collections import Counter
import numpy as np
import os

def create_graph_per_cluster(cluster_name, df_cluster_data, weight_threshold=2):
    """
    Fungsi untuk membuat visualisasi graf yang fokus pada satu klaster spesifik,
    menampilkan entitas penting yang terhubung padanya.
    """
    print(f"\n--- Memproses Graf untuk Klaster: {cluster_name} ---")

    # 1. Bangun struktur nodes dan edges
    ner_cols = ['NER_Person', 'NER_Location', 'NER_Organization', 'NER_Brand', 'NER_Product']
    junk_entities = {'saya', 'kamu', 'dia', 'kita', 'mereka', 'ini', 'itu'}
    nodes = {}
    edges = {}

    # Inisialisasi node topik
    nodes[cluster_name] = {'Type': 'Topik'}

    for _, row in df_cluster_data.iterrows():
        for col in ner_cols:
            ner_type = col.replace('NER_', '')
            ents_string = row[col]
            if pd.isna(ents_string): continue

            entities = [e.strip().lower() for e in str(ents_string).split(';') if e.strip()]

            for ent in entities:
                if len(ent) < 3 or ent in junk_entities: continue

                nodes[ent] = {'Type': ner_type}
                edge_key = tuple(sorted((ent, cluster_name)))
                if edge_key not in edges:
                    edges[edge_key] = 0
                edges[edge_key] += 1

    # 2. Buat Graf NetworkX dengan FILTER
    G = nx.Graph()

    for (source, target), weight in edges.items():
        if weight > weight_threshold:
            G.add_edge(source, target, weight=weight)

    for node_name in G.nodes():
        if node_name in nodes:
            G.nodes[node_name]['type'] = nodes[node_name]['Type']

    if G.number_of_edges() == 0:
        print(f"Tidak ada koneksi yang cukup kuat (di atas {weight_threshold}) untuk klaster '{cluster_name}'.")
        return

    # 3. Visualisasi Graf dengan peningkatan kualitas
    print(f"Memulai visualisasi untuk klaster '{cluster_name}'...")
    plt.figure(figsize=(24, 24))
    
    # Menggunakan layout yang lebih baik untuk visualisasi
    k_value = 1.5 / np.sqrt(G.number_of_nodes()) if G.number_of_nodes() > 0 else 1.5
    pos = nx.spring_layout(G, k=k_value, iterations=100, weight='weight', seed=42)

    # Palet warna yang lebih menarik dan kontras
    color_map = {
        'Topik': '#FF6B6B',      # Merah muda
        'Person': '#4ECDC4',      # Cyan
        'Location': '#45B7D1',    # Biru muda
        'Organization': '#96CEB4', # Hijau muda
        'Brand': '#FFEEAD',       # Kuning muda
        'Product': '#D4A5A5',     # Merah muda pucat
        'Default': '#E9ECEF'      # Abu-abu muda
    }

    # Styling yang lebih baik untuk nodes dan edges
    node_colors = [color_map.get(G.nodes[n].get('type', 'Default'), '#E9ECEF') for n in G.nodes()]
    node_sizes = [20000 if G.nodes[n].get('type') == 'Topik' else G.degree(n) * 800 + 3000 for n in G.nodes()]
    edge_weights = [np.log1p(G[u][v]['weight']) * 2 for u, v in G.edges()]  # Logarithmic scaling for better visibility

    # Menggambar edges dengan alpha yang lebih rendah dan warna yang lebih lembut
    nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.2, edge_color='#666666')
    
    # Menggambar nodes dengan efek shadow
    for n in G.nodes():
        # Draw shadow
        plt.scatter(pos[n][0], pos[n][1], s=node_sizes[list(G.nodes()).index(n)]*1.1,
                   c='#00000022', alpha=0.3)

    # Menggambar nodes utama
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.9)

    # Memperbaiki label dengan font yang lebih baik
    labels_to_draw = {n: n for n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=labels_to_draw, 
                          font_size=14,
                          font_weight='bold',
                          font_family='sans-serif')

    # Memperbaiki judul dan legend
    plt.title(f"Jaringan Entitas untuk Topik:\n{cluster_name}",
              fontsize=36, pad=50, fontweight='bold', fontfamily='sans-serif')
    plt.axis('off')

    # Legend yang lebih informatif
    legend_handles = [plt.Line2D([0], [0], marker='o', color='w', 
                                label=label, markersize=20,
                                markerfacecolor=color,
                                markeredgecolor='white',
                                markeredgewidth=2)
                     for label, color in color_map.items() if label != 'Default']
    
    plt.legend(handles=legend_handles,
              title="Tipe Entitas",
              title_fontsize=16,
              fontsize=14,
              loc='center left',
              bbox_to_anchor=(1, 0.5),
              frameon=True,
              fancybox=True,
              shadow=True)

    # Menyimpan hasil dengan kualitas tinggi
    safe_filename = "".join([c for c in cluster_name if c.isalpha() or c.isdigit() or c==' ']).rstrip().replace(" ", "_")
    output_filename = f'network_cluster_{safe_filename}.png'
    plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    print(f"Visualisasi untuk klaster '{cluster_name}' berhasil disimpan sebagai '{output_filename}'")

In [2]:
# --- FUNGSI UTAMA UNTUK MENJALANKAN SEMUA PROSES ---
def main():
    try:
        # Ganti path jika file Anda berada di lokasi berbeda
        df_ner = pd.read_csv("/content/NER_result_raw.csv")
        df_cluster_sentiment = pd.read_csv("/content/data_semifinal (1).csv")
        df_merged = pd.merge(df_ner, df_cluster_sentiment, on="id", how="inner")

        # Ambil semua nama klaster unik dari data
        unique_clusters = df_merged['topic_label'].dropna().unique()
        print(f"Ditemukan {len(unique_clusters)} klaster unik. Akan dibuat plot untuk masing-masing.")

        # Atur ambang batas kekuatan hubungan di sini
        AMBANG_BATAS_KEKUATAN_HUBUNGAN = 2

        # Loop untuk setiap klaster unik
        for cluster in unique_clusters:
            # Filter data untuk klaster saat ini
            df_current_cluster = df_merged[df_merged['topic_label'] == cluster]
            # Buat visualisasi untuk klaster tersebut
            create_graph_per_cluster(cluster, df_current_cluster, weight_threshold=AMBANG_BATAS_KEKUATAN_HUBUNGAN)

    except FileNotFoundError:
        print("ERROR: Gagal menemukan file. Pastikan 'NER_result_raw.csv' dan 'data_semifinal.csv' berada di folder yang sama.")
    except Exception as e:
        print(f"Terjadi sebuah error: {e}")


In [15]:
df = pd.read_csv("../data/df_final_dashboard_2.csv")
df.head()

,id,Video,Platform,Transcription,Summary,NER_Brand,NER_Location,NER_Organization,NER_Person,NER_Product,Emotion,Topic_ID,Topic_Label,Topic_Keywords,teks_final
0,1,https://drive.google.com/file/d/1uJrAub9Ht_4X_...,Google Drive,5 tenbe ban di rumah untuk lansia yang pertam...,Rencana latihan rumah ini dirancang khusus unt...,NaN,NaN,NaN,NaN,dumbbell,Trust,1,Kebugaran,"latih, otot, kali, gera, pakai",tenbe ban rumah lansia seat stand gerak duduk ...
1,2,https://drive.google.com/file/d/1zM_ds6y4N-22v...,Google Drive,"Abis main badminton, tangan sering sakit. Ema...",Tangan sering mengalami rasa sakit setelah ber...,NaN,NaN,NaN,NaN,Pro8G,Proud,1,Kebugaran,"latih, otot, kali, gera, pakai",habis main badminton tangan sakit gera ulang s...
2,3,https://drive.google.com/file/d/18LgEBVLPh6xc5...,Google Drive,Akhirnya gue ikut event lari lagi setelah ski...,Pengguna mengikuti event lari Dog Fast Run 202...,NaN,Bandung,NaN,Nama gue,Pilkita; grafier mendali,Trust,1,Kebugaran,"latih, otot, kali, gera, pakai",acara lari skip bandung bangun pagi buta acara...
3,4,https://drive.google.com/file/d/1dyMYKBLHBfuAv...,Google Drive,"Bedah strategi main badminton gue, part 1 Nah...",Penulis dan pasangannya fokus pada strategi pe...,NaN,NaN,NaN,Lawan; Lawannya; Vika; Pemain belakang,NaN,Surprise,6,Olahraga,"bola, main, pukul, lawan, kaki",bedah strategi main badminton part angkat bola...
4,5,https://drive.google.com/file/d/1bMMUhXgigGbM5...,Google Drive,"Benda serta igimen Batin Fan gua, part 3 Kali...",Pemain harus menghadapi lawan dengan pertahana...,NaN,NaN,NaN,Batin fan; Lawan; Lawannya; Batin fan,NaN,Proud,6,Olahraga,"bola, main, pukul, lawan, kaki",benda igimen batin fan part kali lawan kuat de...


In [16]:
# Memindahkan 'tasha farasha' dan 'ivan gunawan' dari NER_Brand ke NER_Person, skip jika Brand kosong
def move_brand_to_person(row):
	brand = row['NER_Brand']
	person = row['NER_Person']
	if pd.isna(brand):
		return row
	brand_list = [b.strip() for b in str(brand).split(';') if b.strip()]
	person_list = [p.strip() for p in str(person).split(';') if pd.notna(person) and p.strip()]
	to_move = ['tasha farasha', 'ivan gunawan']
	# Hapus dari brand_list dan tambahkan ke person_list jika belum ada
	for name in to_move:
		if name in [b.lower() for b in brand_list]:
			brand_list = [b for b in brand_list if b.lower() != name]
			if name not in [p.lower() for p in person_list]:
				person_list.append(name)
	row['NER_Brand'] = '; '.join(brand_list) if brand_list else np.nan
	row['NER_Person'] = '; '.join(person_list) if person_list else np.nan
	return row

df = df.apply(move_brand_to_person, axis=1)


In [17]:
print("\nKolom-kolom yang tersedia di dataframe:")
for col in df.columns:
    print(f"- {col}")


Kolom-kolom yang tersedia di dataframe:
- id
- Video
- Platform
- Transcription
- Summary
- NER_Brand
- NER_Location
- NER_Organization
- NER_Person
- NER_Product
- Emotion
- Topic_ID
- Topic_Label
- Topic_Keywords
- teks_final


In [18]:
print("Kolom-kolom yang tersedia:")
print(df.columns.tolist())

Kolom-kolom yang tersedia:
['id', 'Video', 'Platform', 'Transcription', 'Summary', 'NER_Brand', 'NER_Location', 'NER_Organization', 'NER_Person', 'NER_Product', 'Emotion', 'Topic_ID', 'Topic_Label', 'Topic_Keywords', 'teks_final']


In [17]:
df = df.query("Topic_Label == 'Kecantikan'")
df.shape

(215, 15)

In [18]:
# Menggunakan dataframe yang sudah dibaca sebelumnya
df_merged = df.copy()

# Mengatur output path untuk gambar
output_dir = "visualisasi_ner"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Ambil semua nama klaster unik dari data
unique_clusters = df_merged['Topic_Label'].dropna().unique()
print(f"Ditemukan {len(unique_clusters)} klaster unik. Akan dibuat plot untuk masing-masing.")

# Atur ambang batas kekuatan hubungan di sini
AMBANG_BATAS_KEKUATAN_HUBUNGAN = 1  # Menurunkan threshold agar lebih banyak koneksi yang muncul

# Loop untuk setiap klaster unik
for cluster in unique_clusters:
    try:
        # Filter data untuk klaster saat ini
        df_current_cluster = df_merged[df_merged['Topic_Label'] == cluster]
        
        # Jika ada data untuk klaster ini
        if not df_current_cluster.empty:
            # Buat visualisasi untuk klaster tersebut
            create_graph_per_cluster(cluster, df_current_cluster, weight_threshold=AMBANG_BATAS_KEKUATAN_HUBUNGAN)
        else:
            print(f"Tidak ada data untuk klaster: {cluster}")
            
    except Exception as e:
        print(f"Error saat memproses klaster {cluster}: {str(e)}")
        continue  # Lanjut ke klaster berikutnya

Ditemukan 1 klaster unik. Akan dibuat plot untuk masing-masing.

--- Memproses Graf untuk Klaster: Kecantikan ---
Memulai visualisasi untuk klaster 'Kecantikan'...
Visualisasi untuk klaster 'Kecantikan' berhasil disimpan sebagai 'network_cluster_Kecantikan.png'
